# Módulo EP1 · Entrevistas de programación guiadas — Práctica guiada

Los mismos ejercicios que en la web, para trabajar en Colab o en local. **Primero** sigue los pasos; **después** borra tu celda y repítelo de memoria.

In [ ]:
def probar(nombre, casos, entorno=None):
    """Comprueba tu solución. Nunca lanza un error: te dice qué falta."""
    entorno = globals() if entorno is None else entorno
    bien = 0
    for expr, esperado in casos:
        try:
            obtenido = eval(expr, entorno)
            if hasattr(obtenido, "tolist"):
                obtenido = obtenido.tolist()
            ok = obtenido == esperado or (isinstance(esperado, float) and abs(obtenido - esperado) < 1e-9)
        except NameError:
            print(f"PENDIENTE  {nombre}: aún no has escrito esta función")
            return False
        except Exception as e:
            print(f"FALLA      {expr}  ->  {type(e).__name__}: {e}")
            continue
        if ok:
            bien += 1
        else:
            print(f"FALLA      {expr}\n    obtenido: {obtenido!r}\n    esperado: {esperado!r}")
    print(("OK        " if bien == len(casos) else "INCOMPLETO"), f"{nombre}: {bien}/{len(casos)} pruebas")
    return bien == len(casos)

def probar_sql(nombre, consulta, setup, columnas_esperadas, filas_esperadas):
    """Ejecuta tu consulta en una base en memoria y compara el resultado."""
    if not consulta.strip():
        print(f"PENDIENTE  {nombre}: escribe la consulta en la variable `consulta`")
        return False
    entorno = {}
    exec(setup, entorno)
    cur = entorno["con"].execute(consulta)
    columnas, filas = [d[0] for d in cur.description], cur.fetchall()
    ok = columnas == columnas_esperadas and filas == filas_esperadas
    print(("OK        " if ok else "FALLA     "), nombre)
    if not ok:
        print("    obtenido:", columnas, filas)
    return ok

RES = []


### 1. Dos productos que suman el presupuesto
*Tabla hash*

Dada una lista de `precios` y un `objetivo`, devuelve los **índices** `[i, j]` (con `i < j`) de dos productos que sumen
exactamente el objetivo, o `None` si no existen. Es la pregunta de entrevista más conocida.

La solución obvia prueba todos los pares: O(n²). Vamos a bajarla a **O(n)**.


**Paso 1.** Cambia la pregunta: para cada precio, en vez de buscar una pareja, pregúntate **¿he visto ya el precio que me falta?**
Un diccionario responde a eso en un paso. Guardará `precio → índice`.

```python
def dos_sumas(precios, objetivo):
    vistos = {}
```

**Paso 2.** `enumerate` da a la vez la posición y el valor. Calcula lo que falta para llegar al objetivo.

```python
    for i, precio in enumerate(precios):
        falta = objetivo - precio
```

**Paso 3.** Si lo que falta ya está guardado, tienes el par. El índice guardado es menor porque se vio antes: `i < j` se cumple solo.

```python
        if falta in vistos:
            return [vistos[falta], i]
```

**Paso 4.** **Guarda después de comprobar**, no antes: así un producto no se empareja consigo mismo. Si el bucle termina, no hay par.

**Complejidad:** O(n) en tiempo y O(n) en memoria. Has cambiado memoria por velocidad: dilo en voz alta en la entrevista.

```python
        vistos[precio] = i
    return None
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Dos productos que suman el presupuesto', [('dos_sumas([2, 7, 11, 15], 9)', [0, 1]), ('dos_sumas([3, 3], 6)', [0, 1]), ('dos_sumas([1, 2], 7)', None), ('dos_sumas([5], 10)', None)]))

### 2. Primer pedido duplicado
*Conjunto*

`primer_repetido(pedidos)` devuelve el primer identificador que aparece por segunda vez, o `None` si ninguno se repite.


**Paso 1.** Solo necesitas saber si algo **ya apareció**, no cuántas veces ni dónde. Para eso, un conjunto (`set`) es más claro que un diccionario.

```python
def primer_repetido(pedidos):
    vistos = set()
```

**Paso 2.** Recorre en orden: el primero que encuentres repetido es la respuesta y puedes salir ya.

```python
    for pedido in pedidos:
        if pedido in vistos:
            return pedido
```

**Paso 3.** Si no estaba, lo apuntas. Al terminar sin encontrar nada, devuelve `None`.

**Complejidad:** O(n) en tiempo y memoria. Ordenar primero daría O(n log n) y perdería el orden original.

```python
        vistos.add(pedido)
    return None
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Primer pedido duplicado', [('primer_repetido([3, 1, 4, 1, 3])', 1), ('primer_repetido([1, 2, 3])', None), ('primer_repetido([])', None), ("primer_repetido(['a', 'b', 'a'])", 'a')]))

### 3. Paréntesis equilibrados
*Pila*

`equilibrado(texto)` devuelve `True` si todos los `()`, `[]` y `{}` se abren y se cierran en el orden correcto.
Los demás caracteres se ignoran. Un linter o un compilador hacen exactamente esto.


**Paso 1.** Lo último que se abre es lo primero que se cierra. Esa estructura se llama **pila** (*LIFO*), y una lista de Python sirve:
`append` apila y `pop` desapila. El diccionario indica qué apertura corresponde a cada cierre.

```python
def equilibrado(texto):
    pareja = {")": "(", "]": "[", "}": "{"}
    pila = []
```

**Paso 2.** Las aperturas se apilan: son compromisos pendientes de cerrar.

```python
    for c in texto:
        if c in "([{":
            pila.append(c)
```

**Paso 3.** Ante un cierre hay dos fallos posibles: no hay nada abierto, o lo último abierto no coincide. El orden de la condición importa:
`not pila` va primero para no hacer `pop` de una pila vacía.

```python
        elif c in ")]}":
            if not pila or pila.pop() != pareja[c]:
                return False
```

**Paso 4.** Al final no puede quedar nada abierto. Olvidar esta línea es el error típico: `"(("` daría `True`.

**Complejidad:** O(n) en tiempo y memoria.

```python
    return not pila
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Paréntesis equilibrados', [("equilibrado('([]{})')", True), ("equilibrado('(]')", False), ("equilibrado('((')", False), ("equilibrado(')')", False), ("equilibrado('hola (mundo)')", True)]))

### 4. La mejor racha de k días
*Ventana deslizante*

`mejor_racha(ventas, k)` devuelve la suma máxima de `k` días consecutivos. Si hay menos de `k` días, devuelve `None`.

Recalcular cada suma cuesta O(n·k). Con una **ventana deslizante** cuesta O(n).


**Paso 1.** Descarta primero el caso imposible. Comprobar los casos límite al principio es una buena señal en una entrevista.

```python
def mejor_racha(ventas, k):
    if len(ventas) < k:
        return None
```

**Paso 2.** Calcula la primera ventana completa. A partir de aquí no volverás a sumar `k` elementos.

```python
    actual = sum(ventas[:k])
    mejor = actual
```

**Paso 3.** Al mover la ventana un día, **entra uno y sale otro**: el total se actualiza con una resta y una suma, sin recalcular.
`i` es el día que entra y `i - k` el que sale.

```python
    for i in range(k, len(ventas)):
        actual += ventas[i] - ventas[i - k]
        mejor = max(mejor, actual)
```

**Paso 4.** **Complejidad:** O(n) en tiempo y O(1) en memoria. El truco sirve para cualquier problema de «subsecuencia contigua».

```python
    return mejor
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('La mejor racha de k días', [('mejor_racha([1, 5, 2, 3, 7], 2)', 10), ('mejor_racha([4, 4], 2)', 8), ('mejor_racha([1], 2)', None), ('mejor_racha([-1, -2, -3], 1)', -1)]))

### 5. Fusionar dos listas ordenadas
*Dos punteros*

`fusionar(a, b)` recibe dos listas ya ordenadas y devuelve una sola lista ordenada, **sin usar `sorted`**.
Es el paso central de *merge sort*.


**Paso 1.** Un puntero por lista. Como ambas están ordenadas, el siguiente valor siempre es el menor de los dos que señalan.

```python
def fusionar(a, b):
    i, j = 0, 0
    salida = []
```

**Paso 2.** Mientras queden elementos en **las dos**, compara y avanza solo el puntero del que has usado.

```python
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            salida.append(a[i])
            i += 1
        else:
            salida.append(b[j])
            j += 1
```

**Paso 3.** Cuando una se agota, la otra aún puede tener elementos, y ya están ordenados. Olvidar esto es el fallo más habitual.

**Complejidad:** O(n + m) en tiempo y memoria.

```python
    return salida + a[i:] + b[j:]
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Fusionar dos listas ordenadas', [('fusionar([1, 3], [2, 4])', [1, 2, 3, 4]), ('fusionar([], [1])', [1]), ('fusionar([1, 2], [])', [1, 2]), ('fusionar([1, 1], [1])', [1, 1, 1])]))

### 6. Búsqueda binaria
*Búsqueda binaria*

`buscar(lista, objetivo)` recibe una lista **ordenada** y devuelve el índice del objetivo, o `-1` si no está.
Debe hacerlo en O(log n): en un millón de elementos, unas 20 comparaciones.


**Paso 1.** `izq` y `der` delimitan la zona donde aún puede estar. Al principio, toda la lista. Usamos límites **inclusivos**.

```python
def buscar(lista, objetivo):
    izq, der = 0, len(lista) - 1
```

**Paso 2.** La condición es `<=`, no `<`: con `<` no se comprobaría la zona de un solo elemento. `//` es la división entera.

```python
    while izq <= der:
        medio = (izq + der) // 2
```

**Paso 3.** Mira el centro y descarta la mitad donde es imposible que esté. El `+ 1` y el `- 1` son esenciales: sin ellos, el bucle puede no terminar nunca.

```python
        if lista[medio] == objetivo:
            return medio
        elif lista[medio] < objetivo:
            izq = medio + 1
        else:
            der = medio - 1
```

**Paso 4.** Si la zona queda vacía, no estaba.

**Complejidad:** O(log n) en tiempo y O(1) en memoria.

```python
    return -1
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Búsqueda binaria', [('buscar([1, 3, 5, 7], 5)', 2), ('buscar([1, 3, 5, 7], 4)', -1), ('buscar([], 1)', -1), ('buscar([2], 2)', 0), ('buscar([1, 3, 5, 7], 7)', 3)]))

### 7. Los k productos más vendidos
*Conteo y orden*

`top_k(ventas, k)` recibe una lista de nombres de producto y devuelve los `k` más frecuentes, de mayor a menor.
Los empates se resuelven por orden alfabético.


**Paso 1.** `Counter` hace el conteo que harías con un diccionario. En una entrevista, pregunta si puedes usar la biblioteca estándar: casi siempre sí.

```python
from collections import Counter

def top_k(ventas, k):
    conteo = Counter(ventas)
```

**Paso 2.** Ordena con una **clave compuesta**: primero `-frecuencia` (el signo negativo da orden descendente) y después el nombre. Así se resuelven los empates.

```python
    orden = sorted(conteo, key=lambda p: (-conteo[p], p))
```

**Paso 3.** **Complejidad:** O(n + u log u), siendo `u` los productos distintos. Con `heapq.nlargest` bajaría a O(n + u log k), una buena mejora que mencionar.

```python
    return orden[:k]
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Los k productos más vendidos', [("top_k(['te', 'cafe', 'te', 'pan', 'cafe', 'te'], 2)", ['te', 'cafe']), ("top_k(['b', 'a'], 2)", ['a', 'b']), ('top_k([], 3)', []), ("top_k(['x'], 5)", ['x'])]))

### 8. Fusionar turnos solapados
*Intervalos*

`fusionar_turnos(turnos)` recibe pares `[inicio, fin]` en cualquier orden y fusiona los que se solapan o se tocan.
Devuelve la lista ordenada por inicio.


**Paso 1.** La clave es **ordenar por inicio**: así, un intervalo solo puede solaparse con el anterior. Sin ordenar habría que comparar todos con todos.

```python
def fusionar_turnos(turnos):
    salida = []
    for inicio, fin in sorted(turnos):
```

**Paso 2.** Si empieza cuando ya ha terminado el último, no hay solape: abre un intervalo nuevo. Lo guardamos como lista porque puede crecer.

```python
        if not salida or inicio > salida[-1][1]:
            salida.append([inicio, fin])
```

**Paso 3.** Si se solapa, amplía el último. Usa `max`: un turno puede estar **contenido** en el anterior, y `[1, 10]` con `[2, 3]` debe seguir acabando en 10.

**Complejidad:** O(n log n) por la ordenación.

```python
        else:
            salida[-1][1] = max(salida[-1][1], fin)
    return salida
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Fusionar turnos solapados', [('fusionar_turnos([[8, 10], [1, 3], [2, 6]])', [[1, 6], [8, 10]]), ('fusionar_turnos([[1, 4], [4, 5]])', [[1, 5]]), ('fusionar_turnos([[1, 10], [2, 3]])', [[1, 10]]), ('fusionar_turnos([])', [])]))

### 9. La ruta con menos saltos entre almacenes
*Grafos (BFS)*

`saltos(red, origen, destino)` recibe un diccionario `almacén → lista de vecinos` y devuelve el mínimo de saltos entre dos almacenes,
o `-1` si no están conectados. Si origen y destino coinciden, son 0 saltos.


**Paso 1.** La **búsqueda en anchura** explora por capas: primero lo que está a 1 salto, luego a 2... Por eso la primera vez que llega a un nodo, es por el camino más corto.
Usa una cola; `deque` saca por la izquierda en O(1), mientras que `list.pop(0)` es O(n).

```python
from collections import deque

def saltos(red, origen, destino):
    cola = deque([(origen, 0)])
    vistos = {origen}
```

**Paso 2.** Saca el siguiente nodo. Si es el destino, ya tienes la distancia mínima.

```python
    while cola:
        actual, distancia = cola.popleft()
        if actual == destino:
            return distancia
```

**Paso 3.** Añade los vecinos no visitados. **Márcalos al meterlos en la cola**, no al sacarlos, o un mismo nodo entrará varias veces.
`.get(actual, [])` evita un error si un nodo no tiene entrada en el diccionario.

```python
        for vecino in red.get(actual, []):
            if vecino not in vistos:
                vistos.add(vecino)
                cola.append((vecino, distancia + 1))
```

**Paso 4.** Si la cola se vacía, no hay camino.

**Complejidad:** O(V + A) en tiempo, nodos más aristas, y O(V) en memoria.

```python
    return -1
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('La ruta con menos saltos entre almacenes', [("saltos({'A': ['B'], 'B': ['C'], 'C': []}, 'A', 'C')", 2), ("saltos({'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': []}, 'A', 'D')", 2), ("saltos({'A': [], 'B': []}, 'A', 'B')", -1), ("saltos({'A': ['B']}, 'A', 'A')", 0), ("saltos({'A': ['B'], 'B': ['A']}, 'A', 'Z')", -1)]))

### 10. Menos monedas para un importe
*Programación dinámica*

`menos_monedas(monedas, importe)` devuelve el mínimo de monedas que suman exactamente `importe`, o `-1` si es imposible.
Tomar siempre la moneda más grande falla: con `[1, 3, 4]` e importe 6, eso da 4+1+1, pero lo óptimo es 3+3.


**Paso 1.** **Programación dinámica:** resuelve los subproblemas pequeños y reutilízalos. `mejor[x]` será el mínimo de monedas para el importe `x`.
Empieza con un valor imposible, y `mejor[0] = 0`, porque para sumar 0 no hace falta ninguna.

```python
def menos_monedas(monedas, importe):
    imposible = importe + 1
    mejor = [0] + [imposible] * importe
```

**Paso 2.** Para cada importe, prueba cada moneda como **última** moneda usada: quedan `x - moneda` por pagar, y ese subproblema ya está resuelto.

```python
    for x in range(1, importe + 1):
        for moneda in monedas:
            if moneda <= x:
                mejor[x] = min(mejor[x], mejor[x - moneda] + 1)
```

**Paso 3.** Si sigue valiendo el valor imposible, ninguna combinación llega.

**Complejidad:** O(importe × monedas) en tiempo y O(importe) en memoria.

```python
    return mejor[importe] if mejor[importe] != imposible else -1
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
RES.append(probar('Menos monedas para un importe', [('menos_monedas([1, 3, 4], 6)', 2), ('menos_monedas([2], 3)', -1), ('menos_monedas([1, 2, 5], 11)', 3), ('menos_monedas([1], 0)', 0)]))

### 11. SQL: los dos productos con más ingresos por categoría
*SQL · ventanas*

La tabla `ventas(categoria, producto, ingresos)` tiene una fila por producto. Devuelve `categoria, producto, ingresos` con los **2 productos
de más ingresos de cada categoría**, ordenados por categoría y puesto. Es la pregunta de SQL más repetida en entrevistas de datos.


**Paso 1.** `LIMIT 2` no sirve: daría 2 filas en total, no 2 por grupo. Hace falta numerar **dentro de cada categoría**, y eso lo hace una función de ventana.
Empieza una CTE para trabajar por capas.

```sql
WITH clasificadas AS (
  SELECT categoria, producto, ingresos,
```

**Paso 2.** `PARTITION BY` reinicia el contador en cada categoría. `ORDER BY` decide quién es el 1. A diferencia de `GROUP BY`, conserva todas las filas.

```sql
         ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY ingresos DESC) AS puesto
  FROM ventas
)
```

**Paso 3.** Filtra fuera de la CTE: en el `WHERE` de la misma consulta **no** se puede usar una función de ventana, porque aún no se ha calculado.

**Variante que preguntan:** si puede haber empates, usa `DENSE_RANK()`.

```sql
SELECT categoria, producto, ingresos
FROM clasificadas
WHERE puesto <= 2
ORDER BY categoria, puesto
```

In [ ]:
SETUP_11 = 'import sqlite3\ncon = sqlite3.connect(":memory:")\ncon.executescript("""\nCREATE TABLE ventas (categoria TEXT, producto TEXT, ingresos INT);\nINSERT INTO ventas VALUES\n  (\'cafe\', \'molido\', 90), (\'cafe\', \'grano\', 120), (\'cafe\', \'capsulas\', 60),\n  (\'menaje\', \'taza\', 40), (\'menaje\', \'prensa\', 70), (\'te\', \'verde\', 30);\n""")\n'
consulta = """

"""
RES.append(probar_sql('SQL: los dos productos con más ingresos por categoría', consulta, SETUP_11, ['categoria', 'producto', 'ingresos'], [('cafe', 'grano', 120), ('cafe', 'molido', 90), ('menaje', 'prensa', 70), ('menaje', 'taza', 40), ('te', 'verde', 30)]))

### 12. pandas: el cliente que más gasta en cada ciudad
*pandas*

`top_clientes(df)` recibe un DataFrame con columnas `ciudad`, `cliente` e `importe` (una fila por pedido) y devuelve una lista de pares
`[ciudad, cliente]` con el cliente de mayor gasto total de cada ciudad, por orden alfabético de ciudad.


**Paso 1.** Hay dos niveles: un cliente tiene muchos pedidos. Primero **agrega** el gasto por cliente y ciudad. `as_index=False` deja un DataFrame normal.

```python
def top_clientes(df):
    totales = df.groupby(["ciudad", "cliente"], as_index=False)["importe"].sum()
```

**Paso 2.** Ordena por ciudad y por importe descendente. Así, la primera fila de cada ciudad es la del mayor gasto.

```python
    orden = totales.sort_values(["ciudad", "importe"], ascending=[True, False])
```

**Paso 3.** `drop_duplicates` se queda con la primera fila de cada ciudad. También se resuelve con `idxmax`; da igual cuál uses si sabes explicarla.

**Complejidad:** O(n log n) por la ordenación.

```python
    primeros = orden.drop_duplicates("ciudad")
    return primeros[["ciudad", "cliente"]].values.tolist()
```

In [ ]:
# Escribe aquí tu solución


In [ ]:
import pandas as pd
DF = pd.DataFrame({
    "ciudad": ["Sevilla", "Sevilla", "Sevilla", "Cádiz", "Cádiz"],
    "cliente": ["Ana", "Ana", "Luis", "Eva", "Noa"],
    "importe": [30, 40, 50, 10, 20],
})
RES.append(probar('pandas: el cliente que más gasta en cada ciudad', [('top_clientes(DF)', [['Cádiz', 'Noa'], ['Sevilla', 'Ana']])]))

## Resumen

In [ ]:
print(f"Superados: {sum(RES)}/{len(RES)}")
